<a href="https://colab.research.google.com/github/teganmosibineba/transformer-rl-atari/blob/main/transformer_rl_atari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [2]:
pip install transformers==4.40.0 accelerate einops

In [3]:
!pip uninstall -y ale-py shimmy gymnasium gym atari-py 2>/dev/null
!pip install "gymnasium[atari,accept-rom-license]==0.29.1"
!pip install "shimmy[atari]>=0.2.1"

Found existing installation: ale-py 0.10.1
Uninstalling ale-py-0.10.1:
  Successfully uninstalled ale-py-0.10.1
Found existing installation: Shimmy 2.0.0
Uninstalling Shimmy-2.0.0:
  Successfully uninstalled Shimmy-2.0.0
Found existing installation: gymnasium 1.2.3
Uninstalling gymnasium-1.2.3:
  Successfully uninstalled gymnasium-1.2.3
  Using cached gymnasium-0.29.1-py3-none-any.whl.metadata (10 kB)
  Using cached Shimmy-0.2.1-py3-none-any.whl.metadata (2.3 kB)
INFO: pip is looking at multiple versions of shimmy[atari] to determine which version is compatible with other requirements. This could take a while.
  Using cached Shimmy-0.2.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached Shimmy-0.1.0-py3-none-any.whl.metadata (2.1 kB)
ERROR: Cannot install shimmy[atari]==0.1.0, shimmy[atari]==0.2.0 and shimmy[atari]==0.2.1 because these package versions have conflicting dependencies.

The conflict is caused by:
    shimmy[atari] 0.2.1 depends on ale-py~=0.8.1; extra == "atari"
    shimm

In [4]:
!pip uninstall -y ale-py shimmy gymnasium gym 2>/dev/null
!pip install "ale-py==0.10.1"
!pip install "gymnasium[accept-rom-license]==0.29.1"
!pip install "shimmy[atari]>=0.2.1"

Found existing installation: Shimmy 2.0.0
Uninstalling Shimmy-2.0.0:
  Successfully uninstalled Shimmy-2.0.0
Found existing installation: gymnasium 1.2.3
Uninstalling gymnasium-1.2.3:
  Successfully uninstalled gymnasium-1.2.3
  Using cached ale_py-0.10.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.6 kB)
Using cached ale_py-0.10.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, which is not installed.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, which is not installed.
  Using cached gymnasium-0.29.1-py3-none-any.whl.metadata (10 kB)
Using cached gymnasium-0.29.1-py3-none-any.whl (953 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

In [5]:
pip install opencv-python-headless

In [6]:
pip install datasets huggingface_hub

In [7]:
pip install numpy==1.26.4 h5py

In [8]:
pip install wandb tqdm

In [9]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.40.0 accelerate einops
!pip install opencv-python-headless numpy==1.26.4 h5py
!pip install datasets huggingface_hub wandb tqdm

Looking in indexes: https://download.pytorch.org/whl/cu118


In [10]:
import ale_py
import gymnasium as gym

gym.register_envs(ale_py)  # required with ale-py 0.9+

env = gym.make("ALE/Breakout-v5", render_mode=None)
obs, _ = env.reset()
print(obs.shape)  # (210, 160, 3)
env.close()
print("All good")

(210, 160, 3)
All good


In [11]:
!pip install tensorflow tensorflow-datasets

In [12]:
import tensorflow_datasets as tfds
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Google's DQN replay dataset — this is what DT Atari was trained on
# 'mixed' split = roughly 10% of the full 50M replay buffer, manageable on Colab
GAMES = {
    "Breakout": "breakout",
    "Pong":     "pong",
    "Qbert":    "qbert",
}

def load_game_dataset(game_key, split="train[:10%]"):
    ds_name = f"atari/{GAMES[game_key]}"
    ds = tfds.load(
        ds_name,
        split=split,
        shuffle_files=False,
        data_dir="/content/atari_data",   # cache to Colab disk
    )
    return ds

In [13]:
!pip install datasets huggingface_hub

In [14]:
from datasets import load_dataset
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

GAMES = ["atari-breakout", "atari-pong", "atari-qbert"]

# Inspect structure first
ds = load_dataset("jat-project/jat-dataset", "atari-breakout",
                  split="train", trust_remote_code=True)
print(ds[0].keys())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'jat-project/jat-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'jat-project/jat-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be 

dict_keys(['image_observations', 'rewards', 'discrete_actions'])


In [15]:
ep = ds[0]
print(ep.keys())
print()
# Show shape/type of each field
for k, v in ep.items():
    if hasattr(v, '__len__'):
        print(f"{k}: len={len(v)}, type={type(v[0]) if len(v) > 0 else 'empty'}")
    else:
        print(f"{k}: {v}")

dict_keys(['image_observations', 'rewards', 'discrete_actions'])

image_observations: len=3736, type=<class 'PIL.PngImagePlugin.PngImageFile'>
rewards: len=3736, type=<class 'float'>
discrete_actions: len=3736, type=<class 'int'>


In [16]:
def compute_rtg(rewards, gamma=1.0):
    rtg = np.zeros_like(rewards, dtype=np.float32)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        rtg[t] = running
    return rtg


class AtariDTDataset(Dataset):
    def __init__(self, game, context_len=30, split="train"):
        self.K = context_len
        self.segments = []

        ds = load_dataset("jat-project/jat-dataset", game,
                          split=split, trust_remote_code=True)

        for ep in ds:
            obs  = np.array(ep["observations"], dtype=np.float32) / 255.0
            acts = np.array(ep["actions"],      dtype=np.int64)
            rtg  = compute_rtg(np.array(ep["rewards"], dtype=np.float32))
            T    = len(acts)

            # Sliding window with 50% overlap
            step = max(1, context_len // 2)
            for t in range(0, T - context_len, step):
                self.segments.append((
                    obs [t : t + context_len],   # (K,84,84,4)
                    acts[t : t + context_len],   # (K,)
                    rtg [t : t + context_len],   # (K,)
                ))

        print(f"{game}: {len(ds)} episodes → {len(self.segments)} segments")

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        obs, acts, rtg = self.segments[idx]
        return {
            # permute (K,H,W,C) → (K,C,H,W) for PyTorch
            "obs":  torch.tensor(obs).permute(0, 3, 1, 2),
            "acts": torch.tensor(acts),
            "rtg":  torch.tensor(rtg).unsqueeze(-1),         # (K,1)
            "mask": torch.ones(self.K, dtype=torch.long),
        }

In [17]:
# Load without trust_remote_code and inspect keys
from datasets import load_dataset

ds = load_dataset("jat-project/jat-dataset", "atari-breakout", split="train")
ep = ds[0]
print(list(ep.keys()))
for k, v in ep.items():
    print(f"{k}: {type(v).__name__}", end="")
    if hasattr(v, '__len__'):
        print(f", len={len(v)}")
    else:
        print(f", value={v}")

['image_observations', 'rewards', 'discrete_actions']
image_observations: list, len=3736
rewards: list, len=3736
discrete_actions: list, len=3736


In [18]:
import numpy as np

def compute_rtg(rewards: np.ndarray,
                terminals: np.ndarray = None,
                gamma: float = 1.0) -> np.ndarray:
    """
    Compute Return-to-Go for a trajectory.

    Args:
        rewards:   (T,) float array of per-step rewards
        terminals: (T,) bool array — True where an episode ends.
                   Pass None if the array is already a single episode.
        gamma:     discount factor. Use 1.0 for DT (undiscounted).

    Returns:
        rtg: (T,) float array where rtg[t] = sum of rewards[t:]
    """
    T = len(rewards)
    rtg = np.zeros(T, dtype=np.float32)
    running = 0.0

    for t in reversed(range(T)):
        # Reset accumulator at episode boundaries so RTG
        # never leaks across two different episodes
        if terminals is not None and terminals[t]:
            running = 0.0
        running = rewards[t] + gamma * running
        rtg[t] = running

    return rtg

In [19]:
# Vectorised version — faster for large datasets, same result
def compute_rtg_vectorised(rewards: np.ndarray,
                            terminals: np.ndarray = None,
                            gamma: float = 1.0) -> np.ndarray:
    rtg = np.zeros_like(rewards, dtype=np.float32)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        if terminals is not None and terminals[t]:
            running = 0.0
        running = rewards[t] + gamma * running
        rtg[t] = running
    return rtg

In [20]:
# How RTG is used at inference time — this is the key DT trick
def get_target_rtg(game: str) -> float:
    """
    The RTG you condition on at the START of evaluation.
    Set this to the score you WANT the agent to achieve.
    Higher = more ambitious. Must match the scale of your training data.
    """
    # These are roughly 95th-percentile scores from the mixed dataset
    targets = {
        "Breakout": 90.0,   # DQN avg ~30, expert ~800 — aim for ~3x DQN
        "Pong":     20.0,   # Max is 21 — just set to max
        "Qbert":  2500.0,   # DQN avg ~1152 — aim for ~2x
    }
    return targets[game]

# During a rollout, RTG gets decremented by actual rewards received:
def decrement_rtg(rtg_so_far: float, reward_just_received: float) -> float:
    return rtg_so_far - reward_just_received

In [21]:
# Minimal example showing the full RTG lifecycle for one episode
rewards   = np.array([0, 4, 0, 1, 0, 3, 0, 7], dtype=np.float32)
terminals = np.array([0, 0, 0, 0, 0, 0, 0, 1], dtype=bool)

rtg = compute_rtg(rewards, terminals)
print("Rewards:", rewards)
print("RTG:    ", rtg)
# Rewards: [0. 4. 0. 1. 0. 3. 0. 7.]
# RTG:     [15. 15. 11. 11. 10. 10.  7.  7.]

# At t=0: agent is told "you should get 15 more reward from here"
# At t=1: after scoring 4, RTG drops to 11 → "11 more to go"
# At t=7: last step, RTG = the reward itself = 7

Rewards: [0. 4. 0. 1. 0. 3. 0. 7.]
RTG:     [15. 15. 11. 11. 10. 10.  7.  7.]


In [22]:
import torch
import torch.nn as nn
from transformers import GPT2Model, GPT2Config

class DecisionTransformer(nn.Module):
    def __init__(
        self,
        state_dim: int,       # CNN output dim, e.g. 128
        act_dim: int,         # number of discrete actions, e.g. 4 for Breakout
        hidden_size: int = 128,
        context_len: int = 30,
        n_layer: int = 6,
        n_head: int = 8,
        dropout: float = 0.1,
        img_size: int = 84,
        in_channels: int = 4,  # stacked frames
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.context_len = context_len
        self.act_dim = act_dim

        # ── CNN state encoder ──────────────────────────────────────────
        # Processes (B, K, C, H, W) Atari frames → (B, K, state_dim)
        # Architecture mirrors the DQN encoder from the original DT paper
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),  # →(32,20,20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),           # →(64,9,9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),           # →(64,7,7)
            nn.ReLU(),
            nn.Flatten(),                                          # →3136
            nn.Linear(3136, state_dim),
            nn.ReLU(),
        )

        # ── Token embeddings ───────────────────────────────────────────
        # All three token types are projected into the same hidden_size space
        self.embed_state  = nn.Linear(state_dim, hidden_size)
        self.embed_action = nn.Embedding(act_dim, hidden_size)  # discrete actions
        self.embed_rtg    = nn.Linear(1, hidden_size)           # scalar → vector

        # Learned timestep embedding (not positional — DT uses timestep, not position)
        self.embed_timestep = nn.Embedding(10000, hidden_size)

        self.embed_ln = nn.LayerNorm(hidden_size)

        # ── GPT-2 backbone ─────────────────────────────────────────────
        gpt_config = GPT2Config(
            vocab_size=1,            # unused — we bypass the token embedding
            n_embd=hidden_size,
            n_layer=n_layer,
            n_head=n_head,
            n_inner=4 * hidden_size,
            n_positions=3 * context_len,  # 3 tokens per timestep
            resid_pdrop=dropout,
            attn_pdrop=dropout,
            embd_pdrop=dropout,
        )
        self.transformer = GPT2Model(gpt_config)

        # ── Action prediction head ─────────────────────────────────────
        # Applied to the state token output at each timestep
        self.predict_action = nn.Sequential(
    nn.Linear(hidden_size, act_dim),
)
            # No softmax here — use CrossEntropyLoss which expects

In [23]:
# Training step
def training_step(model, batch, optimizer):
    states    = batch["obs"].cuda()      # (B, K, 4, 84, 84)
    actions   = batch["acts"].cuda()     # (B, K)
    rtg       = batch["rtg"].cuda()      # (B, K, 1)
    mask      = batch["mask"].cuda()     # (B, K)
    timesteps = torch.arange(
        states.shape[1], device=states.device
    ).unsqueeze(0).expand(states.shape[0], -1)  # (B, K)

    logits = model(states, actions, rtg, timesteps, mask)
    # (B, K, act_dim) → flatten for CrossEntropy
    loss = nn.CrossEntropyLoss()(
        logits.view(-1, logits.shape[-1]),  # (B*K, act_dim)
        actions.view(-1),                   # (B*K,)
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
    optimizer.step()
    return loss.item()

In [24]:
!pip uninstall torch torchvision -y
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cpu requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
